# TrustGuard Demo

**A Multi-Agent RL Framework for Autonomous Permission Governance in Mobile Ecosystems**

This notebook demonstrates:
1. Generating a synthetic mini-PermissionBench dataset
2. Running the Permission Prediction Model on example apps
3. Simulating one episode of the governance environment
4. Visualising risk scores and enforcement actions


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

from trustguard.models import PermissionPredictionModel, RuntimeRiskEstimator
from trustguard.models.permission_predictor import ANDROID_PERMISSIONS, NUM_PERMISSIONS
from trustguard.environment import PermissionEnv, EnvConfig
from trustguard.agents import MonitoringAgent, RiskAnalysisAgent, EnforcementAgent
from trustguard.agents.policy_networks import ACTION_NO_OP, ACTION_REVOKE
from trustguard.utils.config_utils import seed_everything

seed_everything(42)
device = torch.device('cpu')
print('TrustGuard demo ready.')

## 1. Permission Prediction Model — Example Applications

In [ ]:
# Instantiate a randomly-initialised predictor (use checkpoint for real results)
perm_pred = PermissionPredictionModel(
    embedding_dim=NUM_PERMISSIONS,  # using perm vector as proxy embedding
    num_permissions=NUM_PERMISSIONS,
    hidden_dims=(256, 128),
).eval()

# Example apps: flashlight and messaging
app_names = ['Flashlight', 'Messaging', 'Navigation', 'Malware (covert)']
# Permission vectors (1 = declared, 0 = not declared)
# Indices: CAMERA=5, CONTACTS=6, LOCATION=10, RECORD_AUDIO=12
perm_vectors = torch.zeros(4, NUM_PERMISSIONS)
perm_vectors[0, [26, 27]] = 1.0          # Flashlight: storage only
perm_vectors[1, [5, 6, 12, 21]] = 1.0   # Messaging: camera, contacts, mic, sms
perm_vectors[2, [10, 11]] = 1.0          # Navigation: location
perm_vectors[3, [5, 6, 10, 12, 21, 14]] = 1.0  # Malware: everything

with torch.no_grad():
    risk_scores = perm_pred.risk_scores(perm_vectors).numpy()  # (4, |P|)

# Visualise per-permission risk scores
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
top_perms = ['CAMERA', 'READ_CONTACTS', 'ACCESS_FINE_LOCATION',
             'RECORD_AUDIO', 'READ_PHONE_STATE', 'SEND_SMS',
             'READ_EXTERNAL_STORAGE', 'WRITE_EXTERNAL_STORAGE']
perm_indices = [ANDROID_PERMISSIONS.index(p) for p in top_perms]

for ax, name, scores in zip(axes, app_names, risk_scores):
    sel_scores = scores[perm_indices]
    colors = ['#e74c3c' if s > 0.6 else '#f39c12' if s > 0.3 else '#2ecc71'
              for s in sel_scores]
    bars = ax.barh(top_perms, sel_scores, color=colors)
    ax.set_xlim(0, 1)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xlabel('Risk Score ϱ(p, fᵢ)')

axes[0].set_ylabel('Permission')
fig.suptitle('Per-Permission Risk Scores (ϱ = 1 − p̂)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('Risk scores saved to risk_scores.png')

## 2. Simulation Environment — One Episode

In [ ]:
# Small environment: 10 benign + 3 malicious apps, 200 steps
cfg = EnvConfig(
    num_benign_apps=10,
    num_malicious_apps=3,
    num_permissions=NUM_PERMISSIONS,
    max_steps=200,
    seed=42,
)
env = PermissionEnv(config=cfg, device=device)
obs = env.reset()

risk_history = []
reward_history = []
action_history = []

for step in range(200):
    # Simple rule-based policy for demo (no trained agents needed)
    mean_risk = env.ema_risk.mean().item()
    if mean_risk > 0.5:
        act_enf = ACTION_REVOKE
        perm_tgt = (env.usage_matrix > 0).float()
    else:
        act_enf = ACTION_NO_OP
        perm_tgt = torch.zeros(env.N, env.P)

    obs, reward, done, info = env.step(
        action_monitor=1, action_risk=1,
        action_enforce=act_enf, perm_targets=perm_tgt
    )
    risk_history.append(env.ema_risk.mean().item())
    reward_history.append(reward)
    action_history.append(act_enf)
    if done:
        break

print(f'Episode complete — {step+1} steps')
print(f'Final mean EMA risk: {risk_history[-1]:.4f}')
print(f'Cumulative reward:   {sum(reward_history):.2f}')
print(f'False revocation rate: {env.false_revocation_rate:.4f}')

In [ ]:
# Plot risk and reward over time
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

steps = range(len(risk_history))

# EMA risk
ax1.plot(steps, risk_history, color='#e74c3c', linewidth=1.5)
ax1.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Risk threshold')
ax1.fill_between(steps, risk_history, alpha=0.15, color='#e74c3c')
ax1.set_ylabel('Mean EMA Risk ρ̄', fontsize=10)
ax1.set_ylim(0, 1)
ax1.legend(fontsize=9)
ax1.set_title('TrustGuard Episode Simulation', fontsize=12, fontweight='bold')

# Reward
ax2.plot(steps, reward_history, color='#2ecc71', linewidth=1.2)
ax2.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Step Reward', fontsize=10)

# Enforcement actions
action_colors = ['#3498db' if a == ACTION_NO_OP else '#e74c3c'
                 for a in action_history]
ax3.bar(steps, [1]*len(action_history), color=action_colors, width=1.0)
from matplotlib.patches import Patch
legend = [Patch(facecolor='#3498db', label='no_op'),
          Patch(facecolor='#e74c3c', label='revoke')]
ax3.legend(handles=legend, fontsize=9, loc='upper right')
ax3.set_ylabel('Action', fontsize=10)
ax3.set_xlabel('Governance Timestep', fontsize=10)
ax3.set_yticks([])

plt.tight_layout()
plt.savefig('episode_trace.png', dpi=150, bbox_inches='tight')
plt.show()
print('Episode trace saved to episode_trace.png')

## 3. Load Trained Checkpoint (if available)

In [ ]:
import os

CKPT_PATH = '../outputs/run_001/checkpoint_best.pt'

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    print(f'Checkpoint loaded (iteration {ckpt["iteration"]})')
    print(f'Keys: {list(ckpt.keys())}')
else:
    print(f'No checkpoint found at {CKPT_PATH}.')
    print('Train TrustGuard first: python experiments/train_trustguard.py ...')

---

## Summary

This notebook demonstrates:
- **Permission Prediction Model**: visualises per-permission risk scores for
  flashlight, messaging, navigation, and malware apps.
- **Simulation Environment**: runs a 200-step governance episode with a simple
  rule-based policy and plots the risk/reward/action trace.
- **Checkpoint loading**: shows how to load trained agent weights.

For full training and evaluation, see the experiment scripts in `experiments/`.